In [3]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(r"C:\Users\Asus\Downloads\music_storeanalysis\Chinook_Sqlite.sqlite")
cursor = conn.cursor()
print("Database connected!")

Database connected!


In [4]:
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tables)

             name
0           Album
1          Artist
2        Customer
3        Employee
4           Genre
5         Invoice
6     InvoiceLine
7       MediaType
8        Playlist
9   PlaylistTrack
10          Track


In [5]:
tables_list = ['Album', 'Artist', 'Customer', 'Invoice', 'Track', 'Genre']
for table in tables_list:
    df = pd.read_sql(f"SELECT COUNT(*) as total FROM {table}", conn)
    print(f"{table}: {df['total'][0]} rows")

Album: 347 rows
Artist: 275 rows
Customer: 59 rows
Invoice: 412 rows
Track: 3503 rows
Genre: 25 rows


In [6]:
df1 = pd.read_sql("""
    SELECT c.FirstName || ' ' || c.LastName as customer_name,
           c.Country,
           ROUND(SUM(i.Total), 2) as total_spent
    FROM Customer c
    INNER JOIN Invoice i ON c.CustomerId = i.CustomerId
    GROUP BY c.CustomerId
    ORDER BY total_spent DESC
    LIMIT 5
""", conn)
print("Top 5 Customers by Spending")
print(df1)

Top 5 Customers by Spending
        customer_name         Country  total_spent
0         Helena Holý  Czech Republic        49.62
1  Richard Cunningham             USA        47.62
2          Luis Rojas           Chile        46.62
3     Ladislav Kovács         Hungary        45.62
4       Hugh O'Reilly         Ireland        45.62


In [7]:
df2 = pd.read_sql("""
    SELECT g.Name as genre,
           COUNT(il.TrackId) as total_sales
    FROM Genre g
    INNER JOIN Track t ON g.GenreId = t.GenreId
    INNER JOIN InvoiceLine il ON t.TrackId = il.TrackId
    GROUP BY g.GenreId
    ORDER BY total_sales DESC
    LIMIT 5
""", conn)
print("Top 5 Best Selling Genres")
print(df2)

Top 5 Best Selling Genres
                genre  total_sales
0                Rock          835
1               Latin          386
2               Metal          264
3  Alternative & Punk          244
4                Jazz           80


In [8]:
df3 = pd.read_sql("""
    SELECT ar.Name as artist_name,
           COUNT(il.TrackId) as total_sales
    FROM Artist ar
    INNER JOIN Album al ON ar.ArtistId = al.ArtistId
    INNER JOIN Track t ON al.AlbumId = t.AlbumId
    INNER JOIN InvoiceLine il ON t.TrackId = il.TrackId
    GROUP BY ar.ArtistId
    ORDER BY total_sales DESC
    LIMIT 5
""", conn)
print("Top 5 Artists by Sales")
print(df3)

Top 5 Artists by Sales
               artist_name  total_sales
0              Iron Maiden          140
1                       U2          107
2                Metallica           91
3             Led Zeppelin           87
4  Os Paralamas Do Sucesso           45


In [9]:
df4 = pd.read_sql("""
    SELECT c.Country,
           COUNT(i.InvoiceId) as total_orders,
           ROUND(SUM(i.Total), 2) as total_sales
    FROM Customer c
    INNER JOIN Invoice i ON c.CustomerId = i.CustomerId
    GROUP BY c.Country
    ORDER BY total_sales DESC
    LIMIT 5
""", conn)
print("Top 5 Countries by Sales")
print(df4)

Top 5 Countries by Sales
   Country  total_orders  total_sales
0      USA            91       523.06
1   Canada            56       303.96
2   France            35       195.10
3   Brazil            35       190.10
4  Germany            28       156.48


In [10]:
df5 = pd.read_sql("""
    SELECT t.Name as track_name,
           ar.Name as artist_name,
           COUNT(il.TrackId) as times_sold
    FROM Track t
    INNER JOIN InvoiceLine il ON t.TrackId = il.TrackId
    INNER JOIN Album al ON t.AlbumId = al.AlbumId
    INNER JOIN Artist ar ON al.ArtistId = ar.ArtistId
    GROUP BY t.TrackId
    ORDER BY times_sold DESC
    LIMIT 5
""", conn)
print("Top 5 Best Selling Tracks")
print(df5)

Top 5 Best Selling Tracks
          track_name artist_name  times_sold
0  Balls to the Wall      Accept           2
1   Inject The Venom       AC/DC           2
2         Snowballed       AC/DC           2
3           Overdose       AC/DC           2
4    Deuces Are Wild   Aerosmith           2


In [11]:
conn.close()
print("Database connection closed!")

Database connection closed!
